# Benchmark MDE su DIODE Outdoor — MiDaS v2.1 Small (TFLite)

Notebook di valutazione per modelli di **Monocular Depth Estimation (MDE)** sul subset **outdoor** di DIODE.
Pensato come **template riproducibile**: per testare un altro modello si modifica **solo l'adapter** (Cella 4),
lasciando invariati loader, allineamento e metriche, così che i risultati siano direttamente confrontabili.

**Dataset:** DIODE Outdoor (`val/outdoor` — scene_00022/00023/00024), depth in metri + mask di validità.
**Range di valutazione:** 0.6 m – 20.0 m (parametrizzato, identico per tutti i modelli).
**Internet OFF:** tutto caricato da `/kaggle/input/`.

**Metodologia (identica per ogni modello — requisito di confrontabilità):**
- I modelli MiDaS-like producono **disparità** (inverse depth) affine-invariante → serve allineamento scala+shift.
- Due protocolli di allineamento *per immagine*:
  - **LSQ scale+shift** in spazio disparità (protocollo accademico).
  - **Median scaling** (scala-only) in spazio disparità (protocollo deployment).
- Pixel validi: `mask==1 & MIN ≤ depth ≤ MAX`.
- Metriche: AbsRel, SqRel, RMSE, RMSE_log, δ<1.25, δ<1.25², δ<1.25³ — mediate per immagine.

**Da NON modificare tra notebook diversi** (per restare confrontabili): `MIN_DEPTH`, `MAX_DEPTH`, `N_IMAGES`,
il loader `DIODEOutdoorDataset`, le funzioni di allineamento e `compute_metrics`.
**Da adattare per ogni modello:** la classe adapter (`load`/`preprocess`/`infer`) e l'attributo `output_type`.

---
**Pipeline:**
1. STEP 0 — Ispezione struttura dataset e modello
2. Config (parametri di benchmark fissi + identità modello)
3. `DIODEOutdoorDataset` loader (solo outdoor)
4. Adapter del modello ← **unica parte da cambiare**
5. Allineamento scala+shift (LSQ + Median)
6. Metriche depth standard
7. Loop valutazione + `results_<slug>.json`
8. Tabella riepilogativa

In [ ]:
# ============================================================
# Cella 1 — Import + STEP 0: ispezione struttura
# ============================================================
import os
import glob
import json
import warnings
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tqdm import tqdm

warnings.filterwarnings('ignore')

# --- STEP 0: albero dataset (2-3 livelli) ---
DATASET_ROOT = Path("/kaggle/input/datasets/artemmmtry/diode-a-dense-indoor-and-outdoor-depth-dataset")
OUTDOOR_ROOT = DATASET_ROOT / "val" / "outdoor"

print("=" * 60)
print("STEP 0 — Ispezione struttura dataset")
print("=" * 60)
print(f"\nDataset root esiste: {DATASET_ROOT.exists()}")
print(f"Outdoor root esiste: {OUTDOOR_ROOT.exists()}")

# Albero 3 livelli
print("\n--- Albero dataset (max 3 livelli) ---")
for lvl0 in sorted(DATASET_ROOT.iterdir()):
    print(f"  {lvl0.name}/")
    if lvl0.is_dir():
        for lvl1 in sorted(lvl0.iterdir()):
            print(f"    {lvl1.name}/")
            if lvl1.is_dir():
                for lvl2 in sorted(lvl1.iterdir()):
                    print(f"      {lvl2.name}/")

# Conferma terne .png / _depth.npy / _depth_mask.npy
print("\n--- Verifica terne (png + depth.npy + mask.npy) in OUTDOOR ---")
png_files  = sorted(glob.glob(str(OUTDOOR_ROOT / "**" / "*.png"),            recursive=True))
depth_npy  = sorted(glob.glob(str(OUTDOOR_ROOT / "**" / "*_depth.npy"),      recursive=True))
mask_npy   = sorted(glob.glob(str(OUTDOOR_ROOT / "**" / "*_depth_mask.npy"), recursive=True))
print(f"  .png trovati:            {len(png_files)}")
print(f"  _depth.npy trovati:      {len(depth_npy)}")
print(f"  _depth_mask.npy trovati: {len(mask_npy)}")
if png_files:
    print(f"  Esempio:  {Path(png_files[0]).name}")
    d0 = np.load(depth_npy[0])
    m0 = np.load(mask_npy[0])
    img0 = cv2.imread(png_files[0])
    print(f"  Shape RGB: {img0.shape}")
    print(f"  Shape depth.npy: {d0.shape}  dtype={d0.dtype}  range=[{d0.min():.3f}, {d0.max():.3f}]")
    print(f"  Shape mask.npy:  {m0.shape}  dtype={m0.dtype}  valori unici={np.unique(m0)}")

# Ricerca modello TFLite
print("\n--- Ricerca .tflite su /kaggle/input ---")
tflite_files = glob.glob('/kaggle/input/**/*.tflite', recursive=True)
for f in tflite_files:
    print(f"  {f}")
if not tflite_files:
    print("  ATTENZIONE: nessun .tflite trovato!")

In [ ]:
# ============================================================
# Cella 2 — Configurazione
# ============================================================

# --- Identità del modello (DA CAMBIARE per ogni notebook di confronto) ---
MODEL_NAME = "MiDaS v2.1 small-lite (TFLite)"
MODEL_SLUG = "midas_v21_small"   # usato per il nome di results_<slug>.json

# --- Parametri di benchmark FISSI (devono restare IDENTICI tra i notebook) ---
MIN_DEPTH = 0.6     # metri
MAX_DEPTH = 20.0    # metri
N_IMAGES  = 200     # campione (None = tutto il dataset). Stesso valore per tutti i modelli.

# --- Path modello (specifico del modello) ---
MIDAS_TFLITE_PATH = Path("/kaggle/input/models/intel/midas/tflite/v2-1-small-lite/1/1.tflite")

# --- Path output ---
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"MODEL_NAME:   {MODEL_NAME}")
print(f"MODEL_SLUG:   {MODEL_SLUG}")
print(f"MIN_DEPTH:    {MIN_DEPTH} m")
print(f"MAX_DEPTH:    {MAX_DEPTH} m")
print(f"N_IMAGES:     {N_IMAGES}")
print(f"MiDaS TFLite: {MIDAS_TFLITE_PATH}  —  esiste: {MIDAS_TFLITE_PATH.exists()}")
print(f"Output dir:   {OUTPUT_DIR}")

In [ ]:
# ============================================================
# Cella 3 — DIODEDataset loader (solo outdoor)
# ============================================================

class DIODEOutdoorDataset:
    """
    Carica terne (RGB, depth, mask) dal subset OUTDOOR di DIODE.
    Struttura attesa:
      outdoor/scene_XXXXX/scan_XXXXX/<nome>.png
                                     <nome>_depth.npy
                                     <nome>_depth_mask.npy
    Restituisce:
      rgb   : uint8  (H, W, 3)
      depth : float32 (H, W)  in metri
      mask  : bool   (H, W)   True = pixel valido

    NOTA confrontabilità: l'ordinamento delle terne è deterministico (sorted),
    quindi con lo stesso N_IMAGES tutti i notebook valutano le STESSE immagini.
    """

    def __init__(self, outdoor_root: Path, n: int = None):
        self.samples = self._find_samples(outdoor_root)
        if n is not None:
            self.samples = self.samples[:n]
        print(f"DIODEOutdoorDataset: {len(self.samples)} terne trovate")

    @staticmethod
    def _find_samples(root: Path):
        """
        Raccoglie terne ricorsivamente:
          <name>.png + <name>_depth.npy + <name>_depth_mask.npy
        """
        samples = []
        for png in sorted(root.rglob("*.png")):
            stem  = png.stem
            d_npy = png.parent / f"{stem}_depth.npy"
            m_npy = png.parent / f"{stem}_depth_mask.npy"
            if d_npy.exists() and m_npy.exists():
                samples.append((png, d_npy, m_npy))
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        png_path, d_path, m_path = self.samples[idx]

        # RGB
        img_bgr = cv2.imread(str(png_path))
        if img_bgr is None:
            raise FileNotFoundError(f"Impossibile leggere: {png_path}")
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # Depth (metri) — squeeze se shape (H,W,1)
        depth = np.load(str(d_path)).astype(np.float32)
        if depth.ndim == 3:
            depth = depth.squeeze(axis=-1)

        # Mask — squeeze se shape (H,W,1)
        mask_raw = np.load(str(m_path))
        if mask_raw.ndim == 3:
            mask_raw = mask_raw.squeeze(axis=-1)
        mask = mask_raw.astype(bool)

        return rgb, depth, mask


dataset = DIODEOutdoorDataset(OUTDOOR_ROOT, n=N_IMAGES)

# Verifica campione
rgb0, depth0, mask0 = dataset[0]
print(f"Campione [0]: rgb={rgb0.shape} depth={depth0.shape} mask={mask0.shape}")
print(f"  depth range (tutti pixel): [{depth0.min():.3f}, {depth0.max():.3f}] m")
valid0 = mask0 & (depth0 >= MIN_DEPTH) & (depth0 <= MAX_DEPTH)
print(f"  pixel validi nel range [{MIN_DEPTH},{MAX_DEPTH}] m: {valid0.sum()} / {valid0.size} "
      f"({100*valid0.mean():.1f}%)")

In [ ]:
# ============================================================
# Cella 4 — DepthModelAdapter astratto + MiDaSTFLiteAdapter
# ============================================================
# >>> UNICA PARTE MODEL-SPECIFIC <<<
# Per testare un altro modello MDE: scrivi una nuova classe che eredita da
# DepthModelAdapter, implementa load/preprocess/infer e imposta output_type.
# Tutto il resto del notebook (allineamento, metriche, loop) resta invariato.
from abc import ABC, abstractmethod


class DepthModelAdapter(ABC):
    """
    Interfaccia astratta per adapter di modelli depth.

    output_type indica COSA produce il modello, e determina come l'allineamento
    converte l'output in profondità metrica (vedi Cella 5):
      - "affine_inverse" : disparità / inverse depth affine-invariante (MiDaS, DPT, ...)
      - "metric_depth"   : profondità in metri (ZoeDepth, Metric3D, ...)
      - "affine_depth"   : profondità affine-invariante (scala/shift ignoti)
    """
    output_type = "affine_inverse"

    @abstractmethod
    def load(self):
        """Carica il modello da disco."""

    @abstractmethod
    def preprocess(self, rgb: np.ndarray) -> np.ndarray:
        """RGB uint8 (H,W,3) -> input_tensor pronto per il modello."""

    @abstractmethod
    def infer(self, input_tensor: np.ndarray, target_hw: tuple = None) -> np.ndarray:
        """input_tensor -> mappa grezza (H, W) float32, ridimensionata a target_hw."""


# Costanti preprocessing MiDaS (ImageNet)
_MIDAS_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_MIDAS_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
_MIDAS_INPUT_SIZE = 256   # MiDaS v2.1 small-lite: 256x256


class MiDaSTFLiteAdapter(DepthModelAdapter):
    """
    Adapter per MiDaS v2.1 small in formato TFLite.
    Input:  NHWC float32 normalizzato ImageNet (oppure uint8 se il modello lo richiede).
    Output: disparità (H_orig, W_orig) dopo resize bilineare al GT.
    """
    output_type = "affine_inverse"

    def __init__(self, model_path: Path):
        self.model_path  = str(model_path)
        self.interpreter = None
        self.input_idx   = None
        self.output_idx  = None
        self.input_h     = _MIDAS_INPUT_SIZE
        self.input_w     = _MIDAS_INPUT_SIZE
        self.input_dtype = np.float32

    def load(self):
        self.interpreter = tf.lite.Interpreter(model_path=self.model_path)
        self.interpreter.allocate_tensors()

        in_details  = self.interpreter.get_input_details()
        out_details = self.interpreter.get_output_details()

        self.input_idx   = in_details[0]["index"]
        self.output_idx  = out_details[0]["index"]
        self.input_dtype = in_details[0]["dtype"]

        shape = in_details[0]["shape"]   # (1, H, W, 3)
        self.input_h = int(shape[1])
        self.input_w = int(shape[2])

        print(f"MiDaS TFLite caricato")
        print(f"  Input:  {tuple(shape)}  dtype={self.input_dtype.__name__}")
        print(f"  Output: {tuple(out_details[0]['shape'])}")

    def preprocess(self, rgb: np.ndarray) -> np.ndarray:
        """
        rgb uint8 (H,W,3) -> NHWC tensor per il modello.
        Gestisce sia float32 (normalizzazione ImageNet) sia uint8 (pass-through).
        """
        resized = cv2.resize(rgb, (self.input_w, self.input_h),
                             interpolation=cv2.INTER_LINEAR)
        if self.input_dtype == np.float32:
            tensor = resized.astype(np.float32) / 255.0
            tensor = (tensor - _MIDAS_MEAN) / _MIDAS_STD
        else:
            tensor = resized.astype(self.input_dtype)
        return tensor[np.newaxis]   # NHWC

    def infer(self, input_tensor: np.ndarray, target_hw: tuple = None) -> np.ndarray:
        """
        Esegue inferenza e restituisce la disparità ridimensionata a target_hw (H, W).
        Se target_hw è None, restituisce la risoluzione nativa del modello.
        """
        self.interpreter.set_tensor(self.input_idx, input_tensor)
        self.interpreter.invoke()
        raw = self.interpreter.get_tensor(self.output_idx)  # (1, H_out, W_out) o (1, H, W, 1)

        # Normalizza shape a (H, W)
        raw = raw.squeeze()
        if raw.ndim != 2:
            raise ValueError(f"Output shape inatteso dopo squeeze: {raw.shape}")

        if target_hw is not None:
            raw = cv2.resize(raw, (target_hw[1], target_hw[0]),
                             interpolation=cv2.INTER_LINEAR)
        return raw.astype(np.float32)


adapter = MiDaSTFLiteAdapter(MIDAS_TFLITE_PATH)
adapter.load()

# Test su campione 0
print("\nTest inferenza su campione [0]...")
_inp = adapter.preprocess(rgb0)
_disp = adapter.infer(_inp, target_hw=depth0.shape)
print(f"  output_type: {adapter.output_type}")
print(f"  Output grezzo: shape={_disp.shape}  range=[{_disp.min():.4f}, {_disp.max():.4f}]")

In [ ]:
# ============================================================
# Cella 5 — Allineamento scala+shift: LSQ e Median Scaling
# ============================================================
# Funzioni MODEL-AGNOSTIC: portano qualunque output (disparità o depth) nello
# spazio disparità e applicano lo STESSO allineamento, garantendo metriche
# confrontabili tra modelli diversi. Non modificare tra un notebook e l'altro.

_ALIGN_EPS = 1e-6


def _to_disparity(pred_raw: np.ndarray, output_type: str) -> np.ndarray:
    """
    Porta l'output del modello nello spazio disparità (inverse depth),
    così che LSQ/Median operino in modo identico per ogni modello.
      - affine_inverse : già disparità -> usato direttamente
      - metric_depth / affine_depth : depth (m) -> disparità = 1/depth
    """
    p = pred_raw.astype(np.float64)
    if output_type == "affine_inverse":
        return p
    if output_type in ("metric_depth", "affine_depth"):
        return 1.0 / np.clip(p, _ALIGN_EPS, None)
    raise ValueError(f"output_type non gestito: {output_type}")


def align_lsq(pred_raw, gt_depth, valid, output_type="affine_inverse"):
    """
    Allineamento Least-Squares scale+shift in spazio disparità (protocollo accademico).
    Trova s, t: min ||s * disp + t - (1/gt_depth)||_2  sui pixel validi.
    Restituisce: pred_depth = 1 / clip(s*disp + t, eps, inf)
    """
    disp    = _to_disparity(pred_raw, output_type)
    p       = disp[valid]
    gt_disp = 1.0 / np.clip(gt_depth[valid].astype(np.float64), _ALIGN_EPS, None)

    A = np.stack([p, np.ones_like(p)], axis=1)            # (N, 2)
    (s, t), *_ = np.linalg.lstsq(A, gt_disp, rcond=None)

    aligned_disp = s * disp + t
    pred_depth   = 1.0 / np.clip(aligned_disp, _ALIGN_EPS, None)
    return pred_depth.astype(np.float32), float(s), float(t)


def align_median(pred_raw, gt_depth, valid, output_type="affine_inverse"):
    """
    Allineamento Median Scaling scala-only in spazio disparità (protocollo deployment).
    s = median(1/gt_valid) / median(disp_valid);  pred_depth = 1 / (s * disp)
    """
    disp    = _to_disparity(pred_raw, output_type)
    p       = disp[valid]
    gt_disp = 1.0 / np.clip(gt_depth[valid].astype(np.float64), _ALIGN_EPS, None)

    s = np.median(gt_disp) / (np.median(p) + _ALIGN_EPS)

    aligned_disp = s * disp
    pred_depth   = 1.0 / np.clip(aligned_disp, _ALIGN_EPS, None)
    return pred_depth.astype(np.float32), float(s)


# Test rapido su campione 0
valid0_range = mask0 & (depth0 >= MIN_DEPTH) & (depth0 <= MAX_DEPTH)
pred_lsq0, s0, t0 = align_lsq(_disp, depth0, valid0_range, adapter.output_type)
pred_med0, sm0    = align_median(_disp, depth0, valid0_range, adapter.output_type)
print(f"Campione [0] — LSQ:    s={s0:.4f}  t={t0:.4f}")
print(f"Campione [0] — Median: s={sm0:.4f}")

In [ ]:
# ============================================================
# Cella 6 — compute_metrics
# ============================================================

def compute_metrics(pred_depth: np.ndarray, gt_depth: np.ndarray,
                    valid: np.ndarray,
                    min_d: float = MIN_DEPTH, max_d: float = MAX_DEPTH) -> dict:
    """
    Metriche standard depth sui soli pixel validi nel range [min_d, max_d].
    Predizione e GT vengono clampate a [min_d, max_d] prima del calcolo
    (convenzione standard, es. Monodepth2): limita gli outlier in modo
    identico per tutti i modelli -> metriche confrontabili.

    Metriche:
      AbsRel  = mean(|pred-gt| / gt)
      SqRel   = mean((pred-gt)^2 / gt)
      RMSE    = sqrt(mean((pred-gt)^2))
      RMSE_log= sqrt(mean((log(pred)-log(gt))^2))
      delta1  = mean(max(pred/gt, gt/pred) < 1.25)
      delta2  = mean(max(pred/gt, gt/pred) < 1.25^2)
      delta3  = mean(max(pred/gt, gt/pred) < 1.25^3)
    """
    mask_range = valid & (gt_depth >= min_d) & (gt_depth <= max_d)
    if mask_range.sum() == 0:
        nan = float('nan')
        return dict(abs_rel=nan, sq_rel=nan, rmse=nan, rmse_log=nan,
                    delta1=nan, delta2=nan, delta3=nan, valid_pixels=0)

    gt   = np.clip(gt_depth[mask_range].astype(np.float64),   min_d, max_d)
    pred = np.clip(pred_depth[mask_range].astype(np.float64), min_d, max_d)

    diff     = pred - gt
    abs_rel  = float(np.mean(np.abs(diff) / gt))
    sq_rel   = float(np.mean(diff**2 / gt))
    rmse     = float(np.sqrt(np.mean(diff**2)))
    rmse_log = float(np.sqrt(np.mean((np.log(pred) - np.log(gt))**2)))

    ratio    = np.maximum(pred / gt, gt / pred)
    delta1   = float(np.mean(ratio < 1.25))
    delta2   = float(np.mean(ratio < 1.25**2))
    delta3   = float(np.mean(ratio < 1.25**3))

    return dict(abs_rel=abs_rel, sq_rel=sq_rel, rmse=rmse, rmse_log=rmse_log,
                delta1=delta1, delta2=delta2, delta3=delta3,
                valid_pixels=int(mask_range.sum()))


# Test
m_lsq = compute_metrics(pred_lsq0, depth0, mask0)
m_med = compute_metrics(pred_med0, depth0, mask0)
print("Campione [0] — Metriche LSQ:    ", {k: f"{v:.4f}" if isinstance(v, float) else v
                                           for k, v in m_lsq.items()})
print("Campione [0] — Metriche Median: ", {k: f"{v:.4f}" if isinstance(v, float) else v
                                           for k, v in m_med.items()})

In [ ]:
# ============================================================
# Cella 8 — Loop valutazione principale
# ============================================================

metrics_lsq_accum    = []
metrics_median_accum = []
valid_pixel_ratios   = []
skipped              = 0

for idx in tqdm(range(len(dataset)), desc=f"Valutazione {MODEL_SLUG} su DIODE Outdoor"):
    try:
        rgb, depth, mask = dataset[idx]
    except Exception as e:
        print(f"  Errore lettura campione {idx}: {e}")
        skipped += 1
        continue

    # Maschera valida nel range di valutazione
    valid = mask & (depth >= MIN_DEPTH) & (depth <= MAX_DEPTH)
    valid_pixel_ratios.append(float(valid.mean()))
    if valid.sum() == 0:
        continue

    # Inferenza
    inp  = adapter.preprocess(rgb)
    pred_raw = adapter.infer(inp, target_hw=depth.shape)

    # Allineamento (stesso protocollo per ogni modello, via output_type)
    pred_lsq, _, _ = align_lsq(pred_raw, depth, valid, adapter.output_type)
    pred_med, _    = align_median(pred_raw, depth, valid, adapter.output_type)

    # Metriche
    m_lsq = compute_metrics(pred_lsq, depth, mask)
    m_med = compute_metrics(pred_med, depth, mask)

    if not np.isnan(m_lsq["abs_rel"]):
        metrics_lsq_accum.append(m_lsq)
    if not np.isnan(m_med["abs_rel"]):
        metrics_median_accum.append(m_med)

print(f"\nCampioni processati:  {len(dataset) - skipped}")
print(f"Campioni saltati:     {skipped}")
print(f"Valid pixel ratio medio: {np.mean(valid_pixel_ratios):.3f} "
      f"(range [{np.min(valid_pixel_ratios):.3f}, {np.max(valid_pixel_ratios):.3f}])")


def mean_metrics(accum: list) -> dict:
    """Media delle metriche per immagine su tutti i campioni validi."""
    if not accum:
        return {}
    keys = [k for k in accum[0] if k != "valid_pixels"]
    return {k: float(np.mean([m[k] for m in accum])) for k in keys}


RESULTS_LSQ    = mean_metrics(metrics_lsq_accum)
RESULTS_MEDIAN = mean_metrics(metrics_median_accum)

print("\n--- Metriche medie LSQ ---")
for k, v in RESULTS_LSQ.items():
    print(f"  {k:12s}: {v:.4f}")

print("\n--- Metriche medie Median Scaling ---")
for k, v in RESULTS_MEDIAN.items():
    print(f"  {k:12s}: {v:.4f}")

delta_abs_rel = RESULTS_MEDIAN.get("abs_rel", float('nan')) - RESULTS_LSQ.get("abs_rel", float('nan'))
print(f"\nDelta AbsRel (Median - LSQ): {delta_abs_rel:+.4f}  (costo dello scaling deployment)")

In [ ]:
# ============================================================
# Cella 9 — Salvataggio results_<slug>.json
# ============================================================
# Schema fisso e model-agnostic: un futuro notebook di confronto può caricare
# tutti i results_*.json da /kaggle/working e affiancare i modelli.

results = {
    "model_name":          MODEL_NAME,
    "model_slug":          MODEL_SLUG,
    "output_type":         adapter.output_type,
    "scenario":            "outdoor_diode",
    "depth_range_m":       [MIN_DEPTH, MAX_DEPTH],
    "n_images":            len(dataset),
    "n_evaluated_lsq":     len(metrics_lsq_accum),
    "n_evaluated_median":  len(metrics_median_accum),
    "valid_pixel_ratio":   float(np.mean(valid_pixel_ratios)) if valid_pixel_ratios else 0.0,
    "metrics_lsq":         RESULTS_LSQ,
    "metrics_median":      RESULTS_MEDIAN,
    "delta_abs_rel_median_minus_lsq": float(delta_abs_rel),
    "note":                "Output affine-invariante: allineamento scala+shift obbligatorio prima della valutazione. Metriche mediate per immagine.",
}

results_path = OUTPUT_DIR / f"results_{MODEL_SLUG}.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"results scritto in: {results_path}")

In [ ]:
# ============================================================
# Cella 10 — Tabella riepilogativa finale
# ============================================================
import pandas as pd

rows = []
for proto, res in [("LSQ (accademico)", RESULTS_LSQ),
                   ("Median (deployment)", RESULTS_MEDIAN)]:
    rows.append({
        "Protocollo":  proto,
        "AbsRel":      f"{res.get('abs_rel', float('nan')):.4f}",
        "SqRel":       f"{res.get('sq_rel', float('nan')):.4f}",
        "RMSE":        f"{res.get('rmse', float('nan')):.4f}",
        "RMSE_log":    f"{res.get('rmse_log', float('nan')):.4f}",
        "δ<1.25":      f"{res.get('delta1', float('nan')):.4f}",
        "δ<1.25²":     f"{res.get('delta2', float('nan')):.4f}",
        "δ<1.25³":     f"{res.get('delta3', float('nan')):.4f}",
    })

df_summary = pd.DataFrame(rows).set_index("Protocollo")

print("\n" + "=" * 70)
print(f"{MODEL_NAME} | DIODE Outdoor | Range {MIN_DEPTH}–{MAX_DEPTH} m")
print("=" * 70)
print(df_summary.to_string())
print("=" * 70)
print(f"N immagini valutate:     {len(dataset)}")
print(f"Valid pixel ratio medio: {np.mean(valid_pixel_ratios):.3f}")
print(f"\nDelta AbsRel Median-LSQ: {delta_abs_rel:+.4f}")
print(f"  -> Costo dell'allineamento semplificato (scala-only) rispetto al protocollo accademico (scala+shift)")